In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'DM FSU' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running DM FSU Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'DM FSU 1': 'https://fsu.gov.dm/registered-entities/search-financial-entities',

        }



Typology={

        'DM FSU 1': 'Registered entities',


        }




sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 'RegCtry': [], 'RegCode' : [], 'ListCode': [], 
         'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 'Phone - Mother company': [], 'Check': []}



searchValues = list(string.ascii_lowercase) + list(map(str, range(10)))

now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    
    
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    
    input_element = driver.find_element(By.XPATH, "/html/body/div[1]/div[2]/div/div/input")
    # Enter the search term
    for index, searchValue in enumerate(searchValues):
        # if index ==3:
        #     break
        input_element.send_keys(searchValue)
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        soup2 = BeautifulSoup(driver.page_source, 'html.parser')  
        sleep(2)
        try:
            table = soup2.find('table')
            sleep(2)
            trs = table.find('tbody').find_all('tr')
        except:
            print('No table')
            input_element.clear()
            continue

        
        sleep(2)
        for i,tr in enumerate(trs):
            tds = tr.find_all('td')
            name = tds[0].text
            print(name)
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            
            if len(tds[1].text) > 2 :
                address = tds[1].text
                sleep(1)
                if 'Dominica' in address:
                    country = 'DM'
                    sqldict['Cntry'].append(country)
                if len(address.split(','))>=2:
                    city = address.split(',')[-2]
                    sqldict['City'].append(city)
                    
            else:
                address = ''
            #print(address.strip())
            sqldict['Address_1'].append(address.strip())
            
            if len(tds[2].text) > 2 :
                type = tds[2].text
            else:
                type = ''
            #print(type.strip())
            sqldict['Typology'].append(type.strip())
            
            if len(tds[3].text) >2:
                website = tds[3].text
            else:
                website = ''
            #print(website.strip())
            sqldict['Website'].append(website.strip())
            
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')   
            sqldict = bourange_same_length_array(sqldict)   
            
        input_element.clear()
        
        

    driver.quit()

    


Working with list DM FSU 1
No table
1Click2Go Bank and Trust Ltd
1Click2Go Bank and Trust Ltd
No table
No table
No table
No table
No table
No table
No table


In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_24092\690050426.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:
df.to_csv('with number.csv')